In [ ]:
import sys
sys.path.append('../src')

from linguistic_brain_toolkit.utilities import get_path
PATH = get_path(1)

In [ ]:
import sqlite3
import pandas as pd

from linguistic_brain_toolkit.data import read_data_from_database, load_dataframe_to_sqlite
from linguistic_brain_toolkit.nlp import apply_lemmatization_and_pos_tags
from linguistic_brain_toolkit.utilities import get_path

pd.set_option('display.precision', 15)  # Establecer la precisión a 15 decimales

PATH = get_path(1)
DATABASE_FOLDER = PATH + 'data/processed/transcriptions.db'

df = read_data_from_database(DATABASE_FOLDER, 'nlp_orthographic_transcriptions')

In [ ]:
# Preprocesing
df['name'] = df['orthographic_word'].str.lower()
df['tier'].value_counts()

df_graphemic_transcrip = df.query("tier == 'word'").copy()

# Filtrar el DataFrame para excluir las filas que contienen las cadenas buscadas
strings_to_exclude = ('{', 'sp', 'chuck-e-cheese')

condition = df_graphemic_transcrip.astype(str).apply(
    lambda row: row.str.contains('|'.join(strings_to_exclude)
                                 )).any(axis=1)
df_filtered = df_graphemic_transcrip[~condition].copy()

In [ ]:
# Eliminar articulos y stopwords
import spacy

nlp = spacy.load("en_core_web_lg")

stopwords = nlp.Defaults.stop_words

print(df_filtered.shape)

# 4. Filtro de una sola pasada:
df_filtered = df_filtered.loc[
    ~df_filtered['orthographic_word']
        .str.lower()         # normalizo a minúsculas
        .isin(stopwords)     # membership test vectorizado
].reset_index(drop=True)

print(df_filtered.shape)

In [ ]:
print(df_filtered.shape)
df_filtered = df_filtered.sample(n=5000)
df_filtered.reset_index(drop=True, inplace=True)
print(df_filtered.shape)
df_filtered.head()

# NLP

In [ ]:
# NLP
df_processed = apply_lemmatization_and_pos_tags(df_filtered, 'name', 'large')
# del df, df_graphemic_transcrip, df_filtered  # Delete dataframes to free memory
df_processed.head()

In [ ]:
df_processed['pos_tag_1'].value_counts()

In [ ]:
df_processed['pos_tag_2'].value_counts()

In [ ]:
df_processed['lemmatized_text'].value_counts()

In [ ]:
cols_to_process = [col for col in df_processed.columns if col.startswith('pos_tag_')]

for col in cols_to_process:
    df_processed[col] = df_processed[col].fillna('')  # Manejar valores nulos

df_processed.reset_index(drop=True, inplace=True)
df_processed.drop(['pos_tags'], axis=1, inplace=True)  # Especificar axis=1 para eliminar columnas

# load_dataframe_to_sqlite(
#     df_processed,
#     DATABASE_FOLDER,
#     'nlp_graphemic_transcriptions',
#     if_exists='replace',
#     df_name='df_processed'
#     )

# LLMs

## Personal paralellism

In [ ]:
from langchain_ollama.llms import OllamaLLM

from linguistic_brain_toolkit.llms import LLMParallelProcessor
from linguistic_brain_toolkit.utilities import YamlManager

In [ ]:
# Load prompts
conf_path: str = PATH + 'conf/'
yaml_manager = YamlManager(conf_path + 'prompts.yaml')

prompt_template_grammatical_category = yaml_manager.get_yaml('grammatical_category')
prompt_template_tense_mark = yaml_manager.get_yaml('tense_mark')

In [ ]:
# Approximation with GPT model
df_filtered['grammatical_category_formatted_prompt'] = prompt_template_grammatical_category
del prompt_template_grammatical_category

llm_gpt = OllamaLLM(model='gemma3:12b')

In [ ]:
# # Approximation with reasoning model

# # df_filtered['lexical_field_formatted_prompt'] = lexical_field_prompt_template


# df_filtered['tense_mark_formatted_prompt'] = tense_mark_prompt_template

# llm_rm = OllamaLLM(model='deepseek-r1:8b')

In [ ]:
print(df_filtered.shape)
df_filtered.dropna(subset=['name'])
df_filtered.reset_index(drop=True, inplace=True)
print(df_filtered.shape)
df_filtered.head()

In [ ]:
# print(df_filtered.shape)
# df_filtered = df_filtered.sample(100).copy()

In [ ]:
%%time
processor = LLMParallelProcessor(
    llm=llm_gpt,
    batch_size=2
)


processor.process_batches(
    df=df_filtered,
    data_column='name',  # Columna para formatear el prompt template
    prompt_column='grammatical_category_formatted_prompt',  # Columna con los prompts listos para ser formateados
    result_column='grammatical_category',  # Columna en la que se almacenarán los resultados del LLM
    data_placeholder='word'  # Variable en el prompt template a ser formateada
)

In [ ]:
%%time
processor = LLMParallelProcessor(
    llm=llm_rm,
    batch_size=30
)

# Procesar los lotes
processor.process_batches(
    df=df_filtered,
    data_column='name',  # Columna para formatear el prompt template
    prompt_column='tense_mark_formatted_prompt',  # Columna con los prompts listos para ser formateados
    result_column='tense_mark',  # Columna en la que se almacenarán los resultados del LLM
    data_placeholder='word'  # Variable en el prompt template a ser formateada
)

In [ ]:
df_filtered['grammatical_category'] = df_filtered['grammatical_category'].str.rstrip('.')
df_filtered['grammatical_category'] = df_filtered['grammatical_category'].str.rstrip('\n')
df_filtered['grammatical_category'].value_counts()

In [ ]:
# df_filtered['grammatical_category'] = df_filtered['grammatical_category'].str.rstrip('.')
df_filtered['tense_mark'].value_counts()

In [ ]:
print(df_filtered.loc[124482, 'tense_mark'])

In [ ]:
df_filtered.drop([
    'grammatical_category_formatted_prompt', 'tense_mark_formatted_prompt'
    ], axis=1, inplace=True)  # Especificar axis=1 para eliminar columnas

load_dataframe_to_sqlite(
    df_filtered,
    DATABASE_FOLDER,
    'grammatical_category',
    if_exists='replace',
    df_name='df_filtered'
    )

## LangChain paralellism

In [ ]:
import re

from pydantic import BaseModel, field_validator
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.pydantic import PydanticOutputParser
from langchain_ollama.llms import OllamaLLM

from linguistic_brain_toolkit.utilities import YamlManager


# Load prompts
conf_path: str = PATH + 'conf/'
yaml_manager = YamlManager(conf_path + 'prompts.yaml')

prompt_template_grammatical_category = yaml_manager.get_yaml('grammatical_category')
prompt_template_lexical_field_identifier = yaml_manager.get_yaml('lexical_field_identifier')
prompt_template_tense_mark = yaml_manager.get_yaml('tense_mark')
prompt_template_lexical_field = yaml_manager.get_yaml('lexical_field_adjust')

In [ ]:
df_temporal = df_filtered.copy()

### Grammatical category

In [ ]:
%%time

import re
import pandas as pd
from pydantic import BaseModel, field_validator
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.pydantic import PydanticOutputParser
from langchain_ollama.llms import OllamaLLM

# 1) Función para limpiar bloques <think>…</think>
def strip_think_tags(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# 2) Modelo Pydantic con validador que normaliza a plural
class POSMark(BaseModel):
    grammatical_category: str

    @field_validator('grammatical_category')
    @classmethod
    def normalize_category(cls, v):
        mapping = {
            'noun': 'nouns', 'pronoun': 'pronouns', 'verb': 'verbs',
            'adjective': 'adjectives', 'adverb': 'adverbs',
            'preposition': 'prepositions', 'conjunction': 'conjunctions',
            'interjection': 'interjections', 'article': 'articles',
            'auxiliary verb': 'auxiliary verbs'
        }
        v = strip_think_tags(v.strip().lower())  # normalizamos entrada
        if v in mapping:
            return mapping[v]
        elif v in mapping.values():
            return v
        else:
            raise ValueError(f"Invalid category: {v}")

# 3) Instanciamos modelo y parser
llm = OllamaLLM(model='gemma3:4b')
parser = PydanticOutputParser(pydantic_object=POSMark)

# 4) Prompt reforzado que exige plural
prompt = ChatPromptTemplate.from_template(prompt_template_grammatical_category)

# 5) Cadena base
base_chain = prompt | llm | parser

# 6) Inputs desde DataFrame
words = df_temporal['name'].dropna().tolist()

inputs = [
    {
        "word": w,
        "format_instructions": parser.get_format_instructions()
    }
    for w in words
]

# 7) Llamada batch
raw_results = base_chain.batch(inputs)

# 8) Procesamiento y validación
processed = [
    r if isinstance(r, POSMark) else POSMark.parse_obj(r)
    for r in raw_results
]

# 9) Asignación al DataFrame
df_temporal.loc[df_temporal['name'].notna(), 'grammatical_category'] = [
    p.grammatical_category for p in processed
]

In [ ]:
df_temporal.sample(5)

### Is lexical field

In [ ]:
# Implementar un proceso para que identifique si es un adverbio y si es una palabra que aplica a campo léxico


# 1) Esquema Pydantic para la salida
class LexicalFieldMark(BaseModel):
    is_lexical_field: Literal['yes', 'no']

    @field_validator('is_lexical_field')
    @classmethod
    def normalize_yes_no(cls, v):
        v2 = strip_think_tags(v.strip().lower())
        if v2 in ('yes', 'no'):
            return v2
        raise ValueError(f"Invalid value for is_lexical_field: {v!r}")

# 2) Función para limpiar bloques <think>…</think>
def strip_think_tags(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# 3) Instanciamos modelo y parser
llm = OllamaLLM(model='gemma3:4b')
parser = PydanticOutputParser(pydantic_object=LexicalFieldMark)

# 4) Prompt reforzado que exige plural
prompt = ChatPromptTemplate.from_template(prompt_template_lexical_field_identifier)

# 5) Cadena base
base_chain = prompt | llm | parser

# 6) Inputs desde DataFrame
words = df_temporal['name'].dropna().tolist()

inputs = [
    {
        "word": w,
        "format_instructions": parser.get_format_instructions()
    }
    for w in words
]

# 7) Llamada batch
raw_results = base_chain.batch(inputs)

# 8) Procesamiento y validación
processed = [
    r if isinstance(r, LexicalFieldMark) else LexicalFieldMark.parse_obj(r)
    for r in raw_results
]

# 9) Asignación al DataFrame
df_temporal.loc[df_temporal['name'].notna(), 'is_lexical_field'] = [
    p.is_lexical_field for p in processed
]

In [ ]:
df_temporal.query("is_lexical_field == 'yes'")

In [ ]:
INDEX:int = 19
response:str = df_temporal.loc[INDEX, 'grammatical_category']

print(response)

### Tense

In [ ]:
%%time

# 1) Esquema Pydantic para la salida
class TenseMark(BaseModel):
    tense: Literal['past', 'present', 'future', 'not applicable']
    explanation: str

# 2) Función para limpiar bloques <think>…</think>
def strip_think_tags(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# 3) Instanciamos modelo y parser
llm = OllamaLLM(model='gemma3:4b')  # qwen3:14b
parser = PydanticOutputParser(pydantic_object=TenseMark)

# 4) Definimos el prompt con las instrucciones JSON inyectadas
# prompt = ChatPromptTemplate.from_template(
#     """You are a philologist specializing in current English.
# Your task is to identify whether the word '{word}' contains morphological
# or semantic marks of tense (past, present, future).

# {format_instructions}

# word: {word}"""
# )

prompt = ChatPromptTemplate.from_template(prompt_template_tense_mark)

# 5) Construimos la cadena base: prompt → LLM → parser
base_chain = prompt | llm | parser

# 6) Preparamos el DataFrame y la lista de inputs
# df_temporal = df_filtered.copy()#.head(25)
words = df_temporal['name'].dropna().tolist()

# Cada input debe incluir 'word' y el bloque de instrucciones de formato
inputs = [
    {
        "word": w,
        "format_instructions": parser.get_format_instructions()
    }
    for w in words
]

# 7) Paralelizamos con el método oficial `batch` de LCEL
#    Esto lanza todas las llamadas LLM en paralelo y retorna una lista de TenseMark
raw_results = base_chain.batch(inputs)  # :contentReference[oaicite:0]{index=0}

# 8) Procesamos cada resultado como objeto TenseMark válido
processed = [
    r if isinstance(r, TenseMark) else TenseMark.parse_obj(r)
    for r in raw_results
]

# 9) Extraemos `tense` y `explanation` como columnas separadas
df_temporal.loc[df_temporal['name'].notna(), 'tense'] = [p.tense for p in processed]
df_temporal.loc[df_temporal['name'].notna(), 'tense_explanation'] = [strip_think_tags(p.explanation) for p in processed]

In [ ]:
df_temporal.sample(5)

In [ ]:
df_temporal = df_temporal.query("is_lexical_field == 'yes' and tense != 'not applicable'")

In [ ]:
INDEX:int = 19
response:str = df_temporal.loc[INDEX, 'tense_explanation']

print(response)

In [ ]:
df_temporal.sample(4)
print(df_temporal.query("is_lexical_field == 'yes'").shape)
print(df_temporal.shape)

In [ ]:
df_temporal.head()

In [ ]:
load_dataframe_to_sqlite(
    df_temporal,
    DATABASE_FOLDER,
    'lexical_field_fragmented',
    if_exists='replace',
    df_name='df_temporal'
    )

### Aquí

### Lexical field

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import yaml

from linguistic_brain_toolkit.data import read_data_from_database, load_dataframe_to_sqlite
from linguistic_brain_toolkit.utilities import get_path, YamlManager

In [2]:
from IPython.display import Markdown, display

def mostrar_markdown(texto):
    """
    Muestra texto formateado como Markdown en un notebook de VSCode.
    
    Args:
        texto (str): Cadena en formato Markdown
    """
    display(Markdown(texto))

In [3]:
# Load prompts
pd.set_option('display.precision', 15)  # Establecer la precisión a 15 decimales

PATH = get_path(1)
DATABASE_FOLDER = PATH + 'data/processed/transcriptions.db'

conf_path: str = PATH + 'conf/'
yaml_manager = YamlManager(conf_path + 'prompts.yaml')

# Cargar el archivo YAML
with open(conf_path + 'prompts.yaml', "r", encoding="utf-8") as file:
    data = yaml.safe_load(file)

# Obtener el contenido de la llave "lexical_field_adjust"
prompt_template_lexical_field = data.get("lexical_field_adjust", "")

# Asegurar que la cadena esté en formato UTF-8
prompt_template_lexical_field = prompt_template_lexical_field.encode("utf-8").decode("utf-8")

# print(prompt_template_lexical_field)


from langchain_core.prompts import ChatPromptTemplate

# Crea el prompt sin ejecutarlo
prompt = ChatPromptTemplate.from_template(prompt_template_lexical_field)

# Inspecciona las variables detectadas
print("Variables detectadas:", prompt.input_variables)

Variables detectadas: ['lexical_field']


In [4]:
df_temporal = read_data_from_database(DATABASE_FOLDER, 'lexical_field_fragmented')
# df_temporal = df_temporal.sample(2).copy()
df_temporal.reset_index(drop=True, inplace=True)
df_temporal[['orthographic_word', 'source',
       'lemmatized_text', 'pos_tag_1', 'pos_tag_2', 'name', 'is_lexical_field',
       'tense', 'tense_explanation', 'grammatical_category']].head()

,orthographic_word,source,lemmatized_text,pos_tag_1,pos_tag_2,name,is_lexical_field,tense,tense_explanation,grammatical_category
0,believe,catfishingstrangerstofindmyself,believe,VERB,,believe,yes,present,The word 'believe' carries a present tense mar...,verbs
1,moving,thumbsup,move,VERB,,moving,yes,present,The word 'moving' carries a present tense mark...,adverbs
2,morning,birthofanation,morning,NOUN,,morning,yes,present,The word 'morning' carries a present tense mar...,nouns
3,hour,mybackseatviewofagreatromance,hour,NOUN,,hour,yes,present,The word 'hour' carries a present tense markin...,nouns
4,bring,theshower,bring,VERB,,bring,yes,present,The word 'bring' carries a present tense marki...,verbs


#### Lexical field from words

In [5]:
unique_words = list(df_temporal['lemmatized_text'].unique())
print("Semas para crear campos léxicos:", len(unique_words))

Semas para crear campos léxicos: 357


In [6]:
# from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

template = """Eres un lingüista experto en la gramática del inglés actual. 
  Teniendo en cuenta que entiendes como 'campo léxico' lo siguiente:
  
  "A lexical field is a structured set of words that:
    1. Share one or more **semantic features** (semes) across all members.
    2. Belong to the **same grammatical category** (all nouns, or all adjectives, or all verbs).
    3. As a group, **cover the full scope** of a given conceptual domain.
    4. Exhibit **clearly definable contrasts** by the presence or absence of specific features.
    5. Contain at least one **archiseme**—a feature common to every member of the field." 
  
  y que a partir de semas como 'chair', 'armchair', 'stool', 'bench' y 'sofa' (tienen la misma categoría gramatical: 'sustantivos') puedes construir el campo léxico de los 'asientos'.
  
  Tu tarea consiste en bríndarme la lista de aquellos campos léxicos asociados a tiempo lingüístico (marcas gramáticales asociadas a pasado, presente o futuro) que se puedan crear utilizando como miembros del campo léxico algunos de los semas de la siguiente lista.
  
  lista de semas: {list_of_semes}.

  No debes usar todos los miembros (semas) dados para proponer campos léxicos; basta con tener todos los semas presentes y a partir de ello identificar los posibles campos léxicos, asociado a tiempo lingüístico, que se pueden construir con uno u otro sema como miembro del campo propuesto.
  Si no ves posible proponer algún campo léxico, mencionalo y justifica el porqué.
  Puedes utilizar los mismos semas de la lista de semas en varios campos léxicos si lo ves necesario.

  Tu respuesta debe ser un string que simule una lista de python. Dicha lista estará compuesta por tuplas que a su vez estarán conformadas por lo siguiente:
    1. El nombre del campo léxico propuesto (string).
    2. Una breve descripción del campo léxico propuesto (string).
    3. Una lista con los semas (miembros del campo léxico) que contendría el campo léxico propuesto. Dichos semas deben ser los escogidos de la lista de semas dada inicialmente (lista).
    """

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="gemma3:27b")

chain = prompt | model

response = chain.invoke({"list_of_semes": unique_words})

In [7]:
print(type(response))
print(response)

<class 'str'>
```python
[
    ("Pasado Simple", "Verbos que denotan acciones completadas en un momento específico en el pasado.", ['burn', 'imprison', 'kiss', 'begin', 'remember', 'forget', 'shoot', 'fall', 'want', 'tell', 'see', 'study', 'hurt', 'learn', 'read', 'break', 'offer', 'deliver', 'buy', 'wear', 'try', 'finish', 'live', 'drag', 'decline', 'rush', 'hear', 'dress', 'execute', 'send', 'take', 'get', 'fail', 'eat', 'lie', 'explain', 'wait', 'wonder', 'beat', 'call', 'cook', 'clean', 'crash', 'grow', 'kill', 'find', 'preach', 'swim', 'stab', 'drench', 'dance', 'comprehend', 'punch', 'join', 'lock', 'unlock', 'drift', 'throw', 'wheeze', 'sink', 'fill', 'offend', 'stay', 'close', 'discover', 'hide', 'hold', 'hit', 'count', 'donate', 'cheat', 'invite', 'yell', 'click', 'chase', 'detain', 'drop', 'fire', 'open', 'descend', 'design', 'raise', 'sell', 'repeat', 'confront']),
    ("Futuro Próximo", "Verbos y expresiones que indican una acción que ocurrirá en un futuro cercano.", ['plan'

#### Lexical field proposed to lemmatized words

In [8]:
import pandas as pd
import ast


# 1. Eliminar los fragmentos que no son parte de la lista
limpio = response.strip()

# 2. Eliminar encabezados y markdown
for basura in ["<class 'str'>", '```python', '```']:
    limpio = limpio.replace(basura, '')

# 3. Evaluar el contenido como lista
data_list = ast.literal_eval(limpio)

# 4. Crear el DataFrame
df_lexical_fields = pd.DataFrame(data_list, columns=["lexical_field_proposed", "description", "semes"])

In [9]:
print(len(df_lexical_fields))
df_lexical_fields.head()

10


,lexical_field_proposed,description,semes
0,Pasado Simple,Verbos que denotan acciones completadas en un ...,"[burn, imprison, kiss, begin, remember, forget..."
1,Futuro Próximo,Verbos y expresiones que indican una acción qu...,"[plan, promise, send, deliver, begin, start, t..."
2,Duración Temporal,Sustantivos que expresan periodos de tiempo.,"[hour, day, year, week, month, moment, minute,..."
3,Experiencias Pasadas,Verbos y sustantivos que connotan vivencias o ...,"[remember, forget, experience, dream, past, ch..."
4,Acciones Habituales (Presente),Verbos que describen acciones que se realizan ...,"[work, study, eat, drink, sleep, walk, run, ex..."


In [10]:
from collections import defaultdict

# Paso 1: Diccionario palabra -> lista de (campo, descripción)
word_to_fields = defaultdict(list)
for _, row in df_lexical_fields.iterrows():
    for word in row['semes']:
        word_to_fields[word].append((row['lexical_field_proposed'], row['description']))

# Paso 2: Función que genera filas expandidas
def expand_row_with_lexical_fields(row):
    lemmatized_words = row['lemmatized_text']
    if isinstance(lemmatized_words, str):
        lemmatized_words = lemmatized_words.split()

    matched = []
    for word in lemmatized_words:
        if word in word_to_fields:
            for field, desc in word_to_fields[word]:
                new_row = row.copy()
                new_row['is_in_proposed_lexical_fields'] = True
                new_row['lexical_field_proposed'] = field
                new_row['description'] = desc
                new_row['matched_word'] = word
                matched.append(new_row)
    
    # Si no hay match, agregar una fila con 'not applicable'
    if not matched:
        new_row = row.copy()
        new_row['is_in_proposed_lexical_fields'] = False
        new_row['lexical_field_proposed'] = 'not applicable'
        new_row['description'] = 'not applicable'
        new_row['matched_word'] = None
        matched.append(new_row)

    return matched

# Paso 3: Aplicar la función a todo el DataFrame
expanded_rows = []
for _, row in df_temporal.iterrows():
    expanded_rows.extend(expand_row_with_lexical_fields(row))

# Paso 4: Crear el nuevo DataFrame expandido
df_temporal = pd.DataFrame(expanded_rows)


# AGREGAR EN UNA COLUMNA SI APARECE EN OTROS CAMPOS Y CUALES SON

In [11]:
df_lexical_fields.head(10)

,lexical_field_proposed,description,semes
0,Pasado Simple,Verbos que denotan acciones completadas en un ...,"[burn, imprison, kiss, begin, remember, forget..."
1,Futuro Próximo,Verbos y expresiones que indican una acción qu...,"[plan, promise, send, deliver, begin, start, t..."
2,Duración Temporal,Sustantivos que expresan periodos de tiempo.,"[hour, day, year, week, month, moment, minute,..."
3,Experiencias Pasadas,Verbos y sustantivos que connotan vivencias o ...,"[remember, forget, experience, dream, past, ch..."
4,Acciones Habituales (Presente),Verbos que describen acciones que se realizan ...,"[work, study, eat, drink, sleep, walk, run, ex..."
5,Planificación Futura,Verbos y sustantivos que denotan intenciones y...,"[plan, promise, schedule, arrange, reserve, bo..."
6,Estado Emocional (Presente),Adjetivos y verbos que expresan sentimientos y...,"[happy, sad, angry, afraid, anxious, excited, ..."
7,Acontecimientos Pasados Traumáticos,Verbos y sustantivos asociados a eventos negat...,"[burn, imprison, wound, punishment, battle, ex..."
8,Proceso de Aprendizaje (Pasado),Verbos y sustantivos relacionados con la adqui...,"[learn, study, teach, read, discover, research..."
9,Acciones Repetidas (Presente/Habitual),Verbos que denotan acciones que se realizan de...,"[play, work, study, exercise, cook, clean, eat..."


In [12]:
columns_list=['orthographic_word',
       'lemmatized_text', 'name',
       'is_in_proposed_lexical_fields', 'lexical_field_proposed',
       'description', 'matched_word']

df_temporal[columns_list].head()

,orthographic_word,lemmatized_text,name,is_in_proposed_lexical_fields,lexical_field_proposed,description,matched_word
0,believe,believe,believe,False,not applicable,not applicable,None
1,moving,move,moving,False,not applicable,not applicable,None
2,morning,morning,morning,False,not applicable,not applicable,None
3,hour,hour,hour,True,Duración Temporal,Sustantivos que expresan periodos de tiempo.,hour
4,bring,bring,bring,False,not applicable,not applicable,None


In [ ]:
print('Tamaño de registros antes del filtro:', df_temporal.shape)
df_temporal = df_temporal.query("is_in_proposed_lexical_fields == True")
print('Tamaño de registros antes del filtro:', df_temporal.shape)

Datamño de registro antes de filtro: (1201, 17)
Datamño de registro antes de filtro: (763, 17)


In [14]:
df_temporal[columns_list].head()

,orthographic_word,lemmatized_text,name,is_in_proposed_lexical_fields,lexical_field_proposed,description,matched_word
3,hour,hour,hour,True,Duración Temporal,Sustantivos que expresan periodos de tiempo.,hour
7,started,start,started,True,Futuro Próximo,Verbos y expresiones que indican una acción qu...,start
8,studies,study,studies,True,Pasado Simple,Verbos que denotan acciones completadas en un ...,study
8,studies,study,studies,True,Experiencias Pasadas,Verbos y sustantivos que connotan vivencias o ...,study
8,studies,study,studies,True,Acciones Habituales (Presente),Verbos que describen acciones que se realizan ...,study


In [15]:
load_dataframe_to_sqlite(
    df_temporal,
    DATABASE_FOLDER,
    'proposed_lexical_fields',
    if_exists='replace',
    df_name='df_temporal'
    )

DataFrame df_temporal loaded into table proposed_lexical_fields.


### Lexical field, semes and archisemes

In [ ]:
%%time
from typing import Optional
import re
import pandas as pd
from pydantic import BaseModel, Field, model_validator
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

# 1. Clase LexicalField
class LexicalField(BaseModel):
    field: str
    table: str
    archiseme: Optional[str] = Field(default="")
    distinctive_semantic_features: Optional[str] = Field(default="")
    
    @model_validator(mode='before')
    def handle_missing_fields(cls, data):
        return {**{"archiseme": "", "distinctive_semantic_features": ""}, **data}

# 2. Función de limpieza
def strip_think_tags(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# 3. Suponemos que prompt_template_lexical_field está definido en otro archivo .yml
prompt = ChatPromptTemplate.from_template(prompt_template_lexical_field)

# 4. Instanciación del LLM
llm = OllamaLLM(model='gemma3:4b')

# 5. Cadena resiliente de parsing
def safe_parse(response):
    try:
        if isinstance(response, dict):
            return LexicalField.parse_obj(response)
        elif isinstance(response, str):
            return LexicalField.parse_raw(response)
        return response
    except Exception:
        json_match = re.search(r"\{.*\}", response, re.DOTALL)
        if json_match:
            try:
                return LexicalField.parse_raw(json_match.group())
            except:
                pass
        return LexicalField(field="PARSE_ERROR", table=str(response)[:500])

processing_chain = prompt | llm | safe_parse

# 6. Preparación de datos
# (Se asume que df_temporal ya existe y tiene la columna 'name')
words = df_temporal['name'].dropna().tolist()
inputs = [{"lexical_field": w} for w in words]

# 7. Procesamiento paralelo
processed = processing_chain.batch(inputs)

# 8. Inicialización de columnas en el DataFrame
for col in ['field', 'table', 'archiseme', 'distinctive_semantic_features', 'lexical_field_json']:
    if col not in df_temporal.columns:
        df_temporal[col] = "" if col != 'lexical_field_json' else None

# 9. Asignación segura a DataFrame
mask = df_temporal['name'].notna()
df_temporal.loc[mask, 'field'] = [strip_think_tags(p.field) for p in processed]
df_temporal.loc[mask, 'table'] = [strip_think_tags(p.table) for p in processed]
df_temporal.loc[mask, 'archiseme'] = [strip_think_tags(p.archiseme) for p in processed]
df_temporal.loc[mask, 'distinctive_semantic_features'] = [strip_think_tags(p.distinctive_semantic_features) for p in processed]
df_temporal.loc[mask, 'lexical_field_json'] = [p.json() for p in processed]

In [ ]:
df_temporal[['name', 'field', 'table', 'archiseme', 'distinctive_semantic_features', 'lexical_field_json']].head()

In [ ]:
INDEX:int = 1

field:str = df_temporal.loc[INDEX, 'field']
table:str = df_temporal.loc[INDEX, 'table']
archiseme:str = df_temporal.loc[INDEX, 'archiseme']
distinctive_semantic_features:str = df_temporal.loc[INDEX, 'distinctive_semantic_features']

print('Campo léxico:', field)
mostrar_markdown(table)
print('Archisema(s):', archiseme, '\n')
print('Descripción de sema(s):', distinctive_semantic_features, '\n')

In [ ]:
# %%time
# from typing import Optional
# import re
# import pandas as pd
# from pydantic import BaseModel
# from typing import Optional

# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers.pydantic import PydanticOutputParser
# from langchain_ollama.llms import OllamaLLM

# # 1) Definición actualizada de LexicalField (archiseme y distinctive_semantic_features COMO Optional)
# class LexicalField(BaseModel):
#     field: str
#     table: str
#     archiseme: Optional[str] = ""
#     distinctive_semantic_features: Optional[str] = ""

# # 2) Función para limpiar bloques <think>…</think>
# def strip_think_tags(text: str) -> str:
#     return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# # 3) Instanciamos modelo y parser
# llm = OllamaLLM(model='gemma3:4b')  # modelo local
# parser = PydanticOutputParser(pydantic_object=LexicalField)

# # 4) Inyectamos las instrucciones de formato JSON en el prompt
# prompt_con_instructions = parser.get_format_instructions() + "\n\n" + prompt_template_lexical_field
# prompt = ChatPromptTemplate.from_template(prompt_con_instructions)

# # 5) Construimos la cadena base: prompt → LLM → parser
# base_chain = prompt | llm | parser

# # 6) Preparamos el DataFrame y la lista de inputs
# words = df_temporal['name'].dropna().tolist()

# inputs = [
#     {
#         "lexical_field": w,
#         # Si tu prompt requiere más variables en el dict, agrégalas aquí.
#     }
#     for w in words
# ]

# # 7) Llamadas en paralelo: obtenemos lista de objetos LexicalField (o dicts)
# raw_results = base_chain.batch(inputs)

# # 8) Convertimos cada resultado en objeto LexicalField
# processed = []
# for r in raw_results:
#     if isinstance(r, LexicalField):
#         processed.append(r)
#     else:
#         # Si r es dict o string JSON, usamos parse_obj
#         processed.append(LexicalField.parse_obj(r))

# # 9) Extraemos cada atributo al DataFrame
# mask = df_temporal['name'].notna()

# df_temporal.loc[mask, 'field'] = [strip_think_tags(p.field) for p in processed]
# df_temporal.loc[mask, 'table'] = [strip_think_tags(p.table) for p in processed]
# df_temporal.loc[mask, 'archiseme'] = [strip_think_tags(p.archiseme) for p in processed]
# df_temporal.loc[mask, 'distinctive_semantic_features'] = [
#     strip_think_tags(p.distinctive_semantic_features) for p in processed
# ]

# # 9a) (Opcional) Guardar el JSON completo en otra columna
# df_temporal.loc[mask, 'lexical_field_json'] = [p.json() for p in processed]


In [ ]:
# %%time
# import re
# import pandas as pd
# from pydantic import BaseModel
# from typing import Literal

# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers.pydantic import PydanticOutputParser
# from langchain_ollama.llms import OllamaLLM

# # 1) Esquema Pydantic para la salida
# class LexicalField(BaseModel):
#     field: str
#     table: str
#     archiseme: str
#     distinctive_semantic_features: str

# # 2) Función para limpiar bloques <think>…</think>
# def strip_think_tags(text: str) -> str:
#     return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# # 3) Instanciamos modelo y parser
# llm = OllamaLLM(model='gemma3:4b')  # qwen3:14b
# parser = PydanticOutputParser(pydantic_object=LexicalField)

# # 4) Definimos el prompt con las instrucciones JSON inyectadas
# prompt = ChatPromptTemplate.from_template(prompt_template_lexical_field)

# # 5) Construimos la cadena base: prompt → LLM → parser
# base_chain = prompt | llm | parser

# # 6) Preparamos el DataFrame y la lista de inputs
# df_temporal = df_filtered.copy()#.head(25)
# words = df_temporal['name'].dropna().tolist()

# # Cada input debe incluir 'word' y el bloque de instrucciones de formato
# inputs = [
#     {
#         "lexical_field": w,
#     }
#     for w in words
# ]

# # 7) Paralelizamos con el método oficial `batch` de LCEL
# #    Esto lanza todas las llamadas LLM en paralelo y retorna una lista de LexicalField
# raw_results = base_chain.batch(inputs)  # :contentReference[oaicite:0]{index=0}

# # 8) Procesamos cada resultado como objeto LexicalField válido
# processed = [
#     r if isinstance(r, LexicalField) else LexicalField.parse_obj(r)
#     for r in raw_results
# ]

# # 9) Extraemos `field` y `table` como columnas separadas
# df_temporal.loc[df_temporal['name'].notna(), 'field'] = [p.tense for p in processed]
# df_temporal.loc[df_temporal['name'].notna(), 'table'] = [strip_think_tags(p.explanation) for p in processed]

In [ ]:
# %%time
# import re
# import pandas as pd
# from pydantic import BaseModel
# from typing import Literal

# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers.pydantic import PydanticOutputParser
# from langchain_ollama.llms import OllamaLLM

# # 1) Esquema Pydantic para la salida
# class TenseMark(BaseModel):
#     field: str
#     table: str
#     archiseme: str
#     distinctive_semantic_features: str

# # 2) Función para limpiar bloques <think>…</think>
# def strip_think_tags(text: str) -> str:
#     return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# # 3) Instanciamos modelo y parser
# llm = OllamaLLM(model='gemma3:4b')  # qwen3:14b
# parser = PydanticOutputParser(pydantic_object=TenseMark)

# # 4) Definimos el prompt con las instrucciones JSON inyectadas
# prompt = ChatPromptTemplate.from_template(prompt_template_lexical_field)

# # 5) Construimos la cadena base: prompt → LLM → parser
# base_chain = prompt | llm | parser

# # 6) Preparamos el DataFrame y la lista de inputs
# df_temporal = df_filtered.copy()#.head(25)
# words = df_temporal['name'].dropna().tolist()

# # Cada input debe incluir 'word' y el bloque de instrucciones de formato
# inputs = [
#     {
#         "lexical_field": w,
#     }
#     for w in words
# ]

# # 7) Paralelizamos con el método oficial `batch` de LCEL
# #    Esto lanza todas las llamadas LLM en paralelo y retorna una lista de TenseMark
# raw_results = base_chain.batch(inputs)  # :contentReference[oaicite:0]{index=0}

# # 8) Procesamos cada resultado como objeto TenseMark válido
# processed = [
#     r if isinstance(r, TenseMark) else TenseMark.parse_obj(r)
#     for r in raw_results
# ]

# # 9) Extraemos `tense` y `explanation` como columnas separadas
# df_temporal.loc[df_temporal['name'].notna(), 'tense'] = [p.tense for p in processed]
# df_temporal.loc[df_temporal['name'].notna(), 'tense_explanation'] = [strip_think_tags(p.explanation) for p in processed]

#### Deepseek

In [ ]:
%%time
from typing import Optional
import re
import pandas as pd
from pydantic import BaseModel, Field, model_validator
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

# 1. Clase LexicalField robustecida
class LexicalField(BaseModel):
    field: str
    table: str
    archiseme: Optional[str] = Field(default="")
    distinctive_semantic_features: Optional[str] = Field(default="")
    
    @model_validator(mode='before')
    def handle_missing_fields(cls, data):
        return {**{"archiseme": "", "distinctive_semantic_features": ""}, **data}

# 2. Función de limpieza (mantenida)
def strip_think_tags(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# 3. Modelo y procesamiento
llm = OllamaLLM(model='gemma3:4b')

# 4. Prompt SIN inyección de instrucciones JSON
prompt = ChatPromptTemplate.from_template(prompt_template_lexical_field)  # Asegúrate que tu prompt tiene la instrucción JSON al final

# 5. Cadena resiliente de parsing
def safe_parse(response):
    try:
        if isinstance(response, dict):
            return LexicalField.parse_obj(response)
        elif isinstance(response, str):
            return LexicalField.parse_raw(response)
        return response
    except Exception as e:
        json_match = re.search(r"\{.*\}", response, re.DOTALL)
        if json_match:
            try:
                return LexicalField.parse_raw(json_match.group())
            except:
                pass
        return LexicalField(field="PARSE_ERROR", table=str(response)[:500])

processing_chain = prompt | llm | safe_parse

# 6. Preparación de datos
words = df_temporal['name'].dropna().tolist()
inputs = [{"lexical_field": w} for w in words]

# 7. Procesamiento paralelo
processed = processing_chain.batch(inputs)

# 8. Inicialización de columnas
for col in ['field', 'table', 'archiseme', 'distinctive_semantic_features', 'lexical_field_json']:
    if col not in df_temporal.columns:
        df_temporal[col] = "" if col != 'lexical_field_json' else None

# 9. Asignación segura a DataFrame
mask = df_temporal['name'].notna()
df_temporal.loc[mask, 'field'] = [strip_think_tags(p.field) for p in processed]
df_temporal.loc[mask, 'table'] = [strip_think_tags(p.table) for p in processed]
df_temporal.loc[mask, 'archiseme'] = [strip_think_tags(p.archiseme) for p in processed]
df_temporal.loc[mask, 'distinctive_semantic_features'] = [strip_think_tags(p.distinctive_semantic_features) for p in processed]
df_temporal.loc[mask, 'lexical_field_json'] = [p.json() for p in processed]

In [ ]:
df_temporal[['name', 'field', 'table', 'archiseme', 'distinctive_semantic_features', 'lexical_field_json']].head()

In [ ]:
df_temporal.query("field != 'PARSE_ERROR'")